# Iris 데이터로 XGBoost 분류하기

`Iris` 데이터를 사용해 `XGBoost(XGBClassifier)` 분류 모델을 학습하고 평가

구성:
1. 라이브러리 불러오기
2. Iris 데이터 준비
3. 학습/테스트 데이터 분리
4. XGBoost 모델 학습
5. 정확도, 분류 리포트, 혼동 행렬 확인
6. 특성 중요도 확인
7. 꽃잎 길이/너비 2개 특성으로 결정 경계 시각화

> XGBoost가 설치되어 있지 않다면 터미널 또는 노트북 셀에서 `pip install xgboost`를 먼저 실행하세요.

## `#TODO` 부분에 코드를 채워 넣으세요

In [ ]:
# 필요한 라이브러리를 불러옵니다.
# numpy: 수치 계산
# pandas: 표 형태 데이터 정리
# matplotlib: 그래프 시각화

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn에서 Iris 데이터와 평가 도구를 불러옵니다.
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# XGBoost 분류 모델입니다.
from xgboost import XGBClassifier

In [ ]:
# Iris 데이터를 불러옵니다.
# Iris 데이터는 붓꽃 3종류(setosa, versicolor, virginica)를 분류하는 대표적인 예제 데이터입니다.

iris = load_iris()

# X: 입력 특성입니다.
# 총 4개 특성: sepal length, sepal width, petal length, petal width
X = iris.data

# y: 정답 라벨입니다.
# 0=setosa, 1=versicolor, 2=virginica
y = iris.target

# 특성 이름과 클래스 이름을 따로 저장해두면 해석할 때 편합니다.
feature_names = iris.feature_names
target_names = iris.target_names

# 데이터 형태를 확인합니다.
print("X shape:", X.shape)
print("y shape:", y.shape)
print("클래스 이름:", target_names)

In [ ]:
# 데이터를 pandas DataFrame으로 바꿔서 보기 좋게 확인합니다.
# 이 셀은 모델 학습에는 필수는 아니지만, 데이터 구조를 이해하는 데 도움이 됩니다.

iris_df = pd.DataFrame(X, columns=feature_names)
iris_df["target"] = y
iris_df["target_name"] = [target_names[i] for i in y]

iris_df.head()

In [ ]:
# 학습 데이터와 테스트 데이터를 나눕니다.
# stratify=y는 각 클래스 비율이 학습/테스트 데이터에 비슷하게 유지되도록 합니다.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터 크기:", X_train.shape)
print("테스트 데이터 크기:", X_test.shape)

In [ ]:
# XGBoost 분류 모델을 생성합니다.
#
# n_estimators: 만들 트리의 개수입니다.
# learning_rate: 각 트리가 이전 모델의 오차를 얼마나 강하게 보정할지 정합니다.
# max_depth: 각 트리의 최대 깊이입니다. 값이 커지면 복잡한 모델이 됩니다.
# subsample: 각 트리를 학습할 때 사용할 샘플 비율입니다.
# colsample_bytree: 각 트리를 학습할 때 사용할 특성 비율입니다.
# eval_metric="mlogloss": 다중 분류에서 사용하는 로그 손실 지표입니다.
# tree_method="hist": 빠른 히스토그램 방식으로 트리를 학습합니다.
# n_jobs=1: 일부 환경에서 병렬 처리로 멈추는 문제를 피하기 위해 1개 코어만 사용합니다.
# random_state: 결과 재현을 위해 난수 시드를 고정합니다.

xgb_clf = XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=1,
    random_state=42
)

# 학습 데이터로 XGBoost 모델을 학습합니다.
#TODO

In [ ]:
# 테스트 데이터에 대해 예측합니다.

y_pred = xgb_clf.predict(X_test)

# 정확도는 전체 테스트 데이터 중 몇 개를 맞혔는지 비율로 나타냅니다.
accuracy = accuracy_score(y_test, y_pred)

print("XGBoost Iris 분류 정확도:", accuracy)

In [ ]:
# 클래스별 precision, recall, f1-score를 확인합니다.
#
# precision: 해당 클래스로 예측한 것 중 실제로 맞은 비율
# recall: 실제 해당 클래스 중 모델이 맞힌 비율
# f1-score: precision과 recall의 조화 평균

print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# 혼동 행렬을 시각화합니다.
# 행은 실제 클래스, 열은 예측 클래스를 의미합니다.

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

disp.plot()
plt.title("XGBoost Confusion Matrix - Iris")
plt.show()

In [ ]:
# XGBoost가 어떤 특성을 중요하게 사용했는지 확인합니다.
# feature_importances_ 값이 클수록 모델이 해당 특성을 더 많이 활용했다는 의미입니다.

importances = xgb_clf.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
# 특성 중요도를 막대그래프로 시각화합니다.

plt.figure(figsize=(8, 4))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("XGBoost Feature Importance - Iris")
plt.show()

## 결정 경계 시각화

결정 경계는 2차원 그래프로 그려야 하므로, Iris의 4개 특성 중  
`petal length`와 `petal width` 두 특성만 사용해서 별도의 XGBoost 모델을 학습합니다.

이 부분은 전체 4개 특성 모델과 별개로, **XGBoost가 분류 영역을 어떻게 나누는지 시각적으로 이해하기 위한 코드**입니다.

In [ ]:
# 시각화를 위해 꽃잎 길이와 꽃잎 너비 두 특성만 사용합니다.
# iris.feature_names 기준:
# 2 = petal length (cm)
# 3 = petal width (cm)

X_2d = iris.data[:, [2, 3]]
y_2d = iris.target

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d,
    y_2d,
    test_size=0.2,
    random_state=42,
    stratify=y_2d
)

# 2차원 데이터용 XGBoost 모델입니다.
xgb_clf_2d = XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=1,
    random_state=42
)

xgb_clf_2d.fit(X_train_2d, y_train_2d)

print("2개 특성만 사용한 정확도:", accuracy_score(y_test_2d, xgb_clf_2d.predict(X_test_2d)))

In [ ]:
# 결정 경계를 그리는 함수입니다.
# 2차원 평면의 모든 점에 대해 모델 예측을 수행한 뒤, 예측 클래스를 색으로 표시합니다.

def plot_decision_boundary(model, X, y):
    # 그래프의 x축, y축 범위를 데이터보다 조금 넓게 잡습니다.
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    # 촘촘한 격자 좌표를 만듭니다.
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    # 격자의 모든 점을 모델에 넣어 클래스를 예측합니다.
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points)
    Z = Z.reshape(xx.shape)

    # 예측 결과에 따라 배경색을 칠합니다.
    plt.contourf(xx, yy, Z, alpha=0.3)

    # 실제 데이터를 산점도로 표시합니다.
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k")

    plt.xlabel("Petal length (cm)")
    plt.ylabel("Petal width (cm)")
    plt.title("XGBoost Decision Boundary - Iris")
    plt.show()


plot_decision_boundary(xgb_clf_2d, X_2d, y_2d)